# Preparation Spawn et al. 2020 data

---

In [ ]:
# Directories
dir_data =  '../data/'
dir01 = '../paper_ecosystem_carbon/output/01_data_prep/'

In [ ]:
# Libraries
import os, time, shutil
import numpy as np
import xarray as xr
import rioxarray
from dask.distributed import Lock
from  scipy import ndimage
import matplotlib.pyplot as plt

In [ ]:
# Function to trim memory of workers
def trim_memory() -> int:
    import ctypes
    libc = ctypes.CDLL("libc.so.6")
    return libc.malloc_trim(0)

---

### Pre-processing

In [ ]:
# Initialize dask
from dask_jobqueue import SLURMCluster
cluster = SLURMCluster(queue='compute', 
                       cores=24, #6
                       memory='256 GB',
                       account='bm0891', 
                       interface="ib0",
                       walltime='00:30:00',
                       local_directory='../dask/',
                       job_extra_directives=[
                           '-o ../dask/LOG_worker_%j.o',
                           '-e ../dask/LOG_worker_%j.o'])

# Scale dask clusters
cluster.scale(jobs=5)

# Configurate dashboard url
import dask
dask.config.config.get('distributed').get('dashboard') \
    .update({'link':'{JUPYTERHUB_SERVICE_PREFIX}/proxy/{port}/status'})

# Create client
from dask.distributed import Client
client = Client(cluster)

client

In [ ]:
def prep_spawn():
    """Prepare spawn data"""
    
    def spawn_load(file):
        '''Load spawn files'''

        return  rioxarray.open_rasterio(dir_data + 'spawn2020/' + file, 
                                        chunks=dict(y=5000, x=5000))

    # Get spawn data and extract area
    da_biom_agbc = spawn_load('aboveground_biomass_carbon_2010.tif')
    da_biom_bgbc = spawn_load('belowground_biomass_carbon_2010.tif')
    da_crop_agbc = spawn_load('crop_agbc.tif')
    da_crop_bgbc = spawn_load('crop_bgbc.tif')
    da_gras_agbc = spawn_load('grass_agbc.tif')
    da_gras_bgbc = spawn_load('grass_bgbc.tif')
    
    # Create new Dataset and multiply arrays with 0.1 for convertion to MgC/ha 
    ds = xr.Dataset(dict(band=da_biom_agbc.band, y=da_biom_agbc.y, 
                         x=da_biom_agbc.x))
    ds['spawn_biom_agbc'] = (('band', 'y', 'x'), da_biom_agbc.data * 0.1)
    ds['spawn_biom_bgbc'] = (('band', 'y', 'x'), da_biom_bgbc.data * 0.1)
    ds['spawn_crop_agbc'] = (('band', 'y', 'x'), da_crop_agbc.data * 0.1)
    ds['spawn_crop_bgbc'] = (('band', 'y', 'x'), da_crop_bgbc.data * 0.1)
    ds['spawn_gras_agbc'] = (('band', 'y', 'x'), da_gras_agbc.data * 0.1)
    ds['spawn_gras_bgbc'] = (('band', 'y', 'x'), da_gras_bgbc.data * 0.1)
    
    # Convert to int
    ds = ds.astype(np.float32)
    
    # Add attributes
    ds.attrs = da_biom_agbc.attrs
    
    # Export
    for i in ds.data_vars:
        ds[i].rio.to_raster(dir01 + i + '.tif')
    # Release memory
    client.run(trim_memory);


%time prep_spawn()

In [ ]:
# Close dask cluster
cluster.close()

In [ ]:
# Wait for dask client to completely disconnect
time.sleep(60)

---

### Regridding

In [ ]:
# Import regridding function
from regrid_high_res_v1_01 import regrid_high_res, prep_tif

In [ ]:
def regrid_da(f_source, dir_target, dir_source, dir_out, 
              size_tiles, fill_value=None, olap=1):  
    """Regrid large xarray dataarrays.

    Args:
        f_source (str): The filename (without extension) of the source .tif file to be regridded.
        dir_target (str): Directory containing target grid .tif file.
        dir_source (str): Directory containing the the source  .tif file.
        dir_out (str): Directory to store the output and intermediate files.
        size_tiles (int): Size of the regridding tiles in degrees.
        fill_value (float, optional): Fill value to use in the regridding process. Defaults to None.
        olap (int, optional): Overlap size in degrees for regridding tiles. Defaults to 1.
        
    Returns:
        xarray.Dataset: The combined dataset after regridding.
    """
    # Prepare the target and source data arrays from TIFF files
    da_target = prep_tif(dir_target + 'target_grid_r001.tif', 'target_grid_r001')
    da_source = prep_tif(dir_source + f_source + '.tif', f_source)
    # Regridd source array to target grid
    regrid_high_res(da_target, da_source, dir_out,
                    account='bm0891', partition='shared',
                    size_tiles=size_tiles, olap=olap, fill_value = fill_value,
                    type_export='zarr', del_interm=False)

In [ ]:
# List of files
list_spawn = ['spawn_biom_agbc', 'spawn_biom_bgbc', 'spawn_crop_agbc', 
              'spawn_crop_bgbc', 'spawn_gras_agbc', 'spawn_gras_bgbc']

# Regridding
for i in list_spawn:
    %time regrid_da(i, dir01, dir01, dir01, 20, np.nan, 0.1)
# Takes about 8-12min for one file

---

### Fill land nans

In [ ]:
def fill_nans(var, dir_out):
    """
    Fills land nan and export as zarr
    """

    def fill_nans_array(data, invalid):
        """
        Replace nan of array with nearest value (method: nearest neighbour)
        """
        ind = ndimage.distance_transform_edt(invalid,
                                             return_distances=False,
                                             return_indices=True)
        return data[tuple(ind)]

    # Paths for input and output
    land_mask_path = os.path.join(dir_out, 'ds_prep_copernicus_land_mask_r001.zarr')
    var_data_path = os.path.join(dir_out, f'ds_regridded_{var}.zarr')
    output_path = os.path.join(dir_out, f'ds_prep_{var}_r001.zarr')

    # Read land mask data
    da_land = xr.open_zarr(land_mask_path) \
                .chunk(dict(lat=5000, lon=5000)) \
                .copernicus_land_mask \
                .compute()

    # Read variable data
    da_var = xr.open_zarr(var_data_path)['regridded_' + var]

    # Fill nan using function fill_nans_array
    # If there are no NaNs, skip filling process
    if not da_var.isnull().any():
        da_fill = da_var.values  # No filling required
    else:
        da_fill = fill_nans_array(da_var.values, da_var.isnull().values)

    # Create a new Dataset with filled data
    ds_filled = xr.Dataset(dict(lat = da_var.lat, lon=da_var.lon))
    ds_filled[var] = (('lat', 'lon'), da_fill)

    # Apply land mask to the filled data
    ds_filled = ds_filled.where(da_land)

    # Export the filled dataset to Zarr format
    ds_filled.chunk(dict(lat=5000, lon=5000)) \
             .to_zarr(output_path, mode='w')

# List of files
list_spawn = ['spawn_biom_agbc', 'spawn_biom_bgbc', 'spawn_crop_agbc', 
              'spawn_crop_bgbc', 'spawn_gras_agbc', 'spawn_gras_bgbc']

# Fill nans
for i in list_spawn:
    print(i)
    %time fill_nans(i, dir01)

---

### Post-processing

In [ ]:
xr.open_mfdataset(dir01 + 'ds_prep_spawn_*_*_r001.zarr', 
                  engine='zarr') \
    .to_zarr(dir01 + 'ds_prep_spawn_r001.zarr', mode='w');

In [ ]:
for i in ['spawn_biom_agbc', 'spawn_biom_bgbc', 'spawn_crop_agbc', 
          'spawn_crop_bgbc', 'spawn_gras_agbc', 'spawn_gras_bgbc']:
    shutil.rmtree(dir01 + 'ds_prep_' + i + '_r001.zarr')